# Snowflake notebook MLOps validation

Run in a Snowflake Workspace notebook on Container Runtime (CPU initially).
Use an existing test database, schema, warehouse and SNOWFLAKE_SSE internal stage.
The notebook role needs stage, experiment and model registry access.

Upload the bundled candidate wheel, `wheel_mlops_lifecycle.py`,
`wheel_mlops_platform.py`, `wheel_gpu_smoke.py` and `snowflake.json` to one stage
prefix. Set the prefix below. Downloading does not start training.


In [ ]:
from pathlib import Path
from snowflake.snowpark.context import get_active_session

PAYLOAD_STAGE = "@KUMO_TEST_DB.YOUR_TEST_SCHEMA.YOUR_TEST_STAGE/mlops-payload"
PAYLOAD = Path("/tmp/mlops-payload")
PAYLOAD.mkdir(exist_ok=True)
get_active_session().file.get(PAYLOAD_STAGE, str(PAYLOAD))
list(PAYLOAD.iterdir())


Set the exact wheel filename and version. Run this installation cell once;
restart the notebook kernel afterward, then rerun the download cell and the
configuration cell below. Package downloads require the notebook's approved
external access or preinstalled dependencies. The runner verifies the installed
files against this wheel and runs `pip check`.


In [ ]:
import subprocess
import sys

WHEEL = PAYLOAD / "kumo_engine-CANDIDATE.whl"
subprocess.run([sys.executable, "-m", "pip", "--isolated", "install",
                f"{WHEEL}[snowflake]"], check=True)


Configure existing test resources. Use a new local evidence directory for each
attempt. Each attempt also gets unique remote model, experiment and artifact
names; resources remain available for inspection.


In [ ]:
import importlib.util
import json

WHEEL = PAYLOAD / "kumo_engine-CANDIDATE.whl"
EXPECTED_VERSION = "CANDIDATE"
OUTPUT = Path("/tmp/mlops-evidence/attempt-001")
CONFIG = PAYLOAD / "snowflake.json"
profile = json.loads(CONFIG.read_text())
profile.update(database="KUMO_TEST_DB", schema="YOUR_TEST_SCHEMA",
               warehouse="KUMO_TEST_WH",
               artifact_root="@KUMO_TEST_DB.YOUR_TEST_SCHEMA.YOUR_TEST_STAGE/mlops-validation")
CONFIG.write_text(json.dumps(profile, indent=2))
spec = importlib.util.spec_from_file_location("mlops_platform", PAYLOAD / "wheel_mlops_platform.py")
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
runner.plan(profile)


This cell starts the live test. It trains two versions, restores and scores in
fresh processes, moves the alias, evaluates after labels arrive and checks
provider permission denial when `denied_artifact_root` is configured. Workers
must authenticate independently as the same account, user and role as this
notebook; an unavailable or different identity blocks the run before writes.


In [ ]:
result = runner.run_notebook(config=CONFIG, wheel=WHEEL,
                             expected_version=EXPECTED_VERSION, output_dir=OUTPUT)
display(result)
if result["returncode"] != 0:
    raise RuntimeError("MLOps validation failed or remains blocked; inspect the summary")


Inspect `summary.json`, phase receipts and logs in the evidence directory.
Review the recorded experiment/model identifiers in Snowsight, including epoch
metrics, both versions, alias and monitoring runs. API readback does not qualify
the UI. Exported full epoch history is still a required implementation gap, so
the summary remains blocked even when the implemented phases pass. Retain the
notebook output and evidence before ending its runtime.
